# Launching AWS RDS PostgreSQL

You've gone over the problems with self hosting a database - but how can you start hosting your database on the cloud? Let's work through it together. In this exercise, you will learn how to:

1. Use the AWS Boto3 library (built on the same system as the AWS CLI) to launch and connect to an AWS RDS instance running PostgreSQL
2. Store and retrieve database credentials securely using AWS Secrets Manager
3. Connect to and query your RDS PostgreSQL database
4. Clean up resources to avoid unnecessary charges

These tools, will help you launch, secure, and connect to your database, before shutting it down. Take special care to shut down this database, as if you do not the instance will incur cloud costs out of your budget! 

## Prerequisites

You will need:

- Your Cloud Resources AWS resources with associated Access Key ID, Secret Access Key, and Session Token
- AWS Boto3 installed and configured
- Basic understanding of PostgreSQL and SQL

Your cloud credentials can be found in the workspace under the tab "Cloud Resources".

## Step 1: Install and Configure boto3

First, let's import os, and boto3, the AWS SDK for Python. Make sure you have AWS credentials configured (via the "Cloud Resources" tab in the workspace). 

Copy these somewhere safe. as you will need to paste them into the code segment below. This will be your first task.

## Configure AWS Credentials

#### Task 1: In the cell below, replace the placeholder values with your actual AWS credentials:

**Required:**
- `aws_access_key_id`: Your AWS Access Key ID
- `aws_secret_access_key`: Your AWS Secret Access Key
- `aws_session_token`: Your AWS Session Token

Note, it's usually not best practices to put passwords and access keys and the like into code. Hardcoding is not secure at all - this is only for demonstration purposes!


In [ ]:
import os
import boto3

# Configure AWS credentials - Replace with your actual credentials
session = boto3.Session(
    aws_access_key_id= '### YOUR CODE HERE',
    aws_secret_access_key= '### YOUR CODE HERE',
    aws_session_token= '### YOUR CODE HERE',  # Optional: only needed for temporary credentials
    region_name='us-east-1'  # Change to your preferred region
)

# Create clients using the configured session
sts = session.client('sts')
boto3.setup_default_session(
    aws_access_key_id=session.get_credentials().access_key,
    aws_secret_access_key=session.get_credentials().secret_key,
    aws_session_token=session.get_credentials().token,
    region_name=session.region_name
)

print("Credentials configured! Run the next cell to verify.")

You should see a statement: 'Credentials configured! Run the next cell to verify.' 


You can go ahead and run the next cell.

In [ ]:
# Verify AWS configuration
from botocore.exceptions import ClientError

# Verify AWS configuration
try:
    identity = sts.get_caller_identity()
    # Strip email from ARN and UserId if present
    arn = identity['Arn'].split('=')[0] if '=' in identity['Arn'] else identity['Arn']
    user_id = identity['UserId'].split('=')[0] if '=' in identity['UserId'] else identity['UserId']
    print("✓ AWS Configuration Verified!")
    print(f"Account: {identity['Account']}")
    print(f"User ARN: {arn}")
    print(f"User ID: {user_id}")
except ClientError as e:
    print(f"✗ Error verifying AWS credentials: {e}")
    print("\nPlease check that you've entered valid credentials in the previous cell.")

You should see something like:

✓ AWS Configuration Verified!
Account: (some random account number)

User ARN: arn:aws:sts::(same random account number as above):assumed-role/voclabs/........................

User ID: (random string)

If that's the output you're getting, you're clear to head to the next step.


## Step 2: Create AWS Secrets for Database Credentials

AWS Secrets Manager helps you protect access to your applications, services, and IT resources. Instead of hardcoding credentials in your code, you can store them securely in Secrets Manager. You don't have to do anything here, as we're generating the password using a function provided for you.

Note, it's usually not best practices to print passwords and access keys. This is only for demonstration purposes. 

In [ ]:
import json
import random
import string

# Generate a random password for the database
def generate_password(length=16):
    # AWS RDS doesn't allow: / @ " (space)
    # Using only safe special characters
    characters = string.ascii_letters + string.digits + "!#$%^&*()-_=+"
    return ''.join(random.choice(characters) for i in range(length))

# Database credentials
db_username = "dbadmin"
db_password = generate_password()

# Create a secret structure
secret_dict = {
    "username": db_username,
    "password": db_password
}

print("Generated credentials (save these temporarily):")
print(f"Username: {db_username}")
print(f"Password: {db_password}")

Next, we'll go ahead and put this information into an AWS Secret. AWS Secrets Manager keeps these details in an encrypted, secure vault for us.

In [ ]:
# Store the secret in AWS Secrets Manager using boto3
secret_name = "rds-postgres-credentials"

# Create Secrets Manager client
secretsmanager = boto3.client('secretsmanager')

try:
    response = secretsmanager.create_secret(
        Name=secret_name,
        Description="RDS PostgreSQL credentials for exercise",
        SecretString=json.dumps(secret_dict)
    )
    print("Secret created successfully!")
    print(f"Secret ARN: {response['ARN']}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceExistsException':
        print(f"Secret '{secret_name}' already exists. Updating it instead...")
        response = secretsmanager.update_secret(
            SecretId=secret_name,
            SecretString=json.dumps(secret_dict)
        )
        print(f"Secret updated successfully!")
    else:
        print(f"Error creating secret: {e}")

## Step 3: Prepare for Launch! (of the RDS PostgreSQL Instance)

Now we'll use boto3 to create an RDS instance running PostgreSQL. This process may take 5-10 minutes to complete.

#### Task 2: Go ahead and name the database. You can use the syntax db_name = "....."

You can name it whatever you like!

In [ ]:
# Define RDS instance parameters
db_instance_identifier = "postgres-exercise-db"
db_instance_class = "db.t3.micro"  # Smallest free tier eligible instance
allocated_storage = 20  # GB (minimum for PostgreSQL with gp2 storage)
engine = "postgres"
engine_version = "18.1"

### YOUR CODE HERE

We need to create a subnet for your RDS to run in. A subnet is a network inside a network. Sort of like if you divided your home wifi into two or more pieces; one for smart appliances, and another for work computers or laptops. 

In [ ]:
# Query available subnets and create DB Subnet Group
# RDS requires a DB Subnet Group when not using the default setup

# Create EC2 client to query VPC and subnets
ec2 = boto3.client('ec2')

# Get default VPC
vpcs_response = ec2.describe_vpcs(Filters=[{'Name': 'isDefault', 'Values': ['true']}])
if vpcs_response['Vpcs']:
    default_vpc_id = vpcs_response['Vpcs'][0]['VpcId']
    print(f"Default VPC: {default_vpc_id}")
else:
    # If no default VPC, get the first available VPC
    vpcs_response = ec2.describe_vpcs()
    if vpcs_response['Vpcs']:
        default_vpc_id = vpcs_response['Vpcs'][0]['VpcId']
        print(f"Using VPC: {default_vpc_id}")
    else:
        raise Exception("No VPC found. Please create a VPC first.")

# Get subnets in the VPC
subnets_response = ec2.describe_subnets(Filters=[{'Name': 'vpc-id', 'Values': [default_vpc_id]}])
subnets = subnets_response['Subnets']

if not subnets:
    raise Exception("No subnets found in VPC. Please create subnets first.")

print(f"\nFound {len(subnets)} subnet(s):")
for subnet in subnets:
    print(f"  - Subnet ID: {subnet['SubnetId']}, AZ: {subnet['AvailabilityZone']}, CIDR: {subnet['CidrBlock']}")

# Create DB Subnet Group
db_subnet_group_name = "rds-exercise-subnet-group"
# Use all available subnets
subnet_ids = [subnet['SubnetId'] for subnet in subnets]

try:
    rds_client = boto3.client('rds')
    subnet_group_response = rds_client.create_db_subnet_group(
        DBSubnetGroupName=db_subnet_group_name,
        DBSubnetGroupDescription="Subnet group for RDS exercise",
        SubnetIds=subnet_ids
    )
    print(f"\nDB Subnet Group '{db_subnet_group_name}' created successfully!")
    print(f"Using subnet(s): {', '.join(subnet_ids)}")
except ClientError as e:
    if e.response['Error']['Code'] == 'DBSubnetGroupAlreadyExists':
        print(f"\nDB Subnet Group '{db_subnet_group_name}' already exists. Will use existing group.")
    else:
        raise e

Next, we have to create a security group that allows connections into the database.

In [ ]:
# Create a security group that allows PostgreSQL connections
security_group_name = "rds-postgres-sg"

try:
    # Create security group
    sg_response = ec2.create_security_group(
        GroupName=security_group_name,
        Description="Security group for PostgreSQL RDS instance",
        VpcId=default_vpc_id
    )
    security_group_id = sg_response['GroupId']
    print(f"Security Group created: {security_group_id}")
    
    # Add inbound rule for PostgreSQL (port 5432)
    ec2.authorize_security_group_ingress(
        GroupId=security_group_id,
        IpPermissions=[
            {
                'IpProtocol': 'tcp',
                'FromPort': 5432,
                'ToPort': 5432,
                'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'Allow PostgreSQL from anywhere'}]
            }
        ]
    )
    print("Inbound rule added: Allow port 5432 from anywhere")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'InvalidGroup.Duplicate':
        # Security group already exists, get its ID
        sg_response = ec2.describe_security_groups(
            Filters=[
                {'Name': 'group-name', 'Values': [security_group_name]},
                {'Name': 'vpc-id', 'Values': [default_vpc_id]}
            ]
        )
        security_group_id = sg_response['SecurityGroups'][0]['GroupId']
        print(f"Security Group '{security_group_name}' already exists: {security_group_id}")
    else:
        raise e

Finally, we'll define some helper functions - that will help us get information about it once it's launched.

In [ ]:
# Define helper functions for RDS management
import time

def check_rds_status(db_identifier):
    """Check the status of the RDS instance"""
    try:
        response = rds.describe_db_instances(DBInstanceIdentifier=db_identifier)
        status = response['DBInstances'][0]['DBInstanceStatus']
        return status
    except ClientError as e:
        print(f"Error checking RDS status: {e}")
        return None

def wait_for_rds_available(db_identifier, max_wait_time=600, check_interval=30):
    """Wait for the RDS instance to become available"""
    elapsed_time = 0
    print("Waiting for RDS instance to become available...")
    
    while elapsed_time < max_wait_time:
        status = check_rds_status(db_identifier)
        print(f"Status after {elapsed_time}s: {status}")
        
        if status == "available":
            print("RDS instance is now available!")
            return True
        
        time.sleep(check_interval)
        elapsed_time += check_interval
    
    print("Timeout waiting for RDS instance.")
    return False

def get_rds_endpoint(db_identifier):
    """Get the endpoint address of the RDS instance"""
    try:
        response = rds.describe_db_instances(DBInstanceIdentifier=db_identifier)
        endpoint = response['DBInstances'][0]['Endpoint']['Address']
        port = response['DBInstances'][0]['Endpoint']['Port']
        return endpoint, port
    except (ClientError, KeyError) as e:
        print(f"Error retrieving endpoint: {e}")
        return None, None

def get_secret(secret_name):
    """Retrieve secret from AWS Secrets Manager"""
    try:
        response = secretsmanager.get_secret_value(SecretId=secret_name)
        secret_string = json.loads(response['SecretString'])
        return secret_string
    except ClientError as e:
        print(f"Error retrieving secret: {e}")
        return None

print("Helper functions defined successfully!")

Before we can connect securely, we need to download the AWS RDS certificate bundle for SSL verification. This is so we can have a fully encrypted connection - but you don't need to worry about configuring this, as we've done it for you.

In [ ]:
import urllib.request
import os

# Download AWS RDS certificate bundle
cert_url = "https://truststore.pki.rds.amazonaws.com/global/global-bundle.pem"
cert_path = "global-bundle.pem"

try:
    print(f"Downloading AWS RDS certificate bundle from {cert_url}...")
    urllib.request.urlretrieve(cert_url, cert_path)
    print(f"Certificate downloaded successfully to: {os.path.abspath(cert_path)}")
except Exception as e:
    print(f"Error downloading certificate: {e}")
    raise

## Step 4: Prepare a Query!

Let's prepare a query for our Postgres instance. 

#### Task 3: As part of the exercise, let's create a very simple query. 

You can use SELECT version(); or you can even use something like SELECT 123; - the option is entirely up to you, but make sure it's the simplest query you can think of.

Assign the query string to the variable ```query_text``` (i.e. your solution could be ```query_text = "SELECT version();"``` )

In [2]:
### YOUR CODE HERE

### Step 5: Create the RDS Instance, Wait for Availability, and Test Connection

The following cell will:
1. Create the RDS PostgreSQL instance
2. Wait for it to become available (this can take 5-10 minutes)
3. Retrieve the endpoint address
4. Get credentials from AWS Secrets Manager
5. Connect to the database using psycopg2
6. Run a simple query to verify the connection works

This is a comprehensive cell that handles the entire workflow from creation to verification.

In [ ]:
# Create RDS client
rds = boto3.client('rds')

# Import psycopg2 for PostgreSQL connections
import psycopg2

try:
    # Step 1: Create the RDS instance
    response = rds.create_db_instance(
        DBInstanceIdentifier=db_instance_identifier,
        DBInstanceClass=db_instance_class,
        Engine=engine,
        EngineVersion=engine_version,
        MasterUsername=db_username,
        MasterUserPassword=db_password,
        AllocatedStorage=allocated_storage,
        DBName=db_name,
        DBSubnetGroupName=db_subnet_group_name,
        VpcSecurityGroupIds=[security_group_id],
        PubliclyAccessible=True,
        BackupRetentionPeriod=0,
        MultiAZ=False,
        StorageType='gp2'
    )
    print("RDS instance creation initiated successfully!")
    print(f"DB Instance: {response['DBInstance']['DBInstanceIdentifier']}")
    print(f"Status: {response['DBInstance']['DBInstanceStatus']}")
    
    # Step 2: Wait for the instance to become available
    if wait_for_rds_available(db_instance_identifier):
        
        # Step 3: Get the endpoint
        endpoint, port = get_rds_endpoint(db_instance_identifier)
        print(f"\nRDS Endpoint: {endpoint}")
        print(f"Port: {port}")
        
        # Step 4: Get credentials from Secrets Manager
        secret_data = get_secret(secret_name)
        if secret_data:
            # Step 5: Connect to the database with SSL verification
            print("\nConnecting to the database...")
            try:
                conn = psycopg2.connect(
                    host=endpoint,
                    port=port,
                    database=db_name,
                    user=secret_data['username'],
                    password=secret_data['password'],
                    sslmode='verify-full',  # Verify SSL certificate
                    sslrootcert=cert_path,  # Path to AWS RDS certificate bundle
                    connect_timeout=10
                )
                
                # Step 6: Run a simple query to verify connection
                cur = conn.cursor()
                cur.execute(query_text)
                result = cur.fetchone()
                print(f"Successfully connected to PostgreSQL!")
                print(f"Query result: {result[0]}")
                
                cur.close()
                conn.close()
                print("Connection closed.")
            except Exception as e:
                print(f"Database connection error: {e}")
                raise
        else:
            print("Failed to retrieve credentials from Secrets Manager.")
    else:
        print("RDS instance did not become available within the timeout period.")
        
except ClientError as e:
    if e.response['Error']['Code'] == 'DBInstanceAlreadyExists':
        print(f"DB instance '{db_instance_identifier}' already exists.")
    else:
        print(f"Error creating RDS instance: {e}")
except Exception as e:
    print(f"Error during RDS setup and connection: {e}")

## Step 6: Clean Up Resources

**IMPORTANT:** To avoid unnecessary AWS charges, make sure to clean up the resources you created in this exercise.

In [ ]:
# Close the database connection
try:
    if connection:
        if 'cursor' in locals():
            cursor.close()
        connection.close()
        print("PostgreSQL connection closed.")
    else:
        print("No connection to close.")
except Exception as e:
    print(f"Error closing connection: {e}")

In [ ]:
# Delete the RDS instance
try:
    response = rds.delete_db_instance(
        DBInstanceIdentifier=db_instance_identifier,
        SkipFinalSnapshot=True
    )
    print("RDS instance deletion initiated successfully!")
    print(f"DB Instance: {response['DBInstance']['DBInstanceIdentifier']}")
    print(f"Status: {response['DBInstance']['DBInstanceStatus']}")
except ClientError as e:
    print(f"Error deleting RDS instance: {e}")

In [ ]:
# Wait for the RDS instance to be fully deleted
def wait_for_rds_deleted(db_identifier, max_wait_time=600, check_interval=30):
    """Wait for the RDS instance to be completely deleted"""
    elapsed_time = 0
    print("Waiting for RDS instance to be deleted...")
    
    while elapsed_time < max_wait_time:
        try:
            response = rds.describe_db_instances(DBInstanceIdentifier=db_identifier)
            status = response['DBInstances'][0]['DBInstanceStatus']
            print(f"Status after {elapsed_time}s: {status}")
        except ClientError as e:
            if e.response['Error']['Code'] == 'DBInstanceNotFound':
                print(f"Status after {elapsed_time}s: Instance not found - deletion complete!")
                return True
            else:
                print(f"Error checking status: {e}")
                return False
        
        time.sleep(check_interval)
        elapsed_time += check_interval
    
    print("Timeout waiting for RDS instance deletion.")
    return False

# Wait for deletion to complete
wait_for_rds_deleted(db_instance_identifier)

Once the RDS instance is deleted, we can clean up the remaining AWS resources (Secret, Subnet Group, and Security Group).

In [ ]:
# Delete the secret from AWS Secrets Manager
try:
    response = secretsmanager.delete_secret(
        SecretId=secret_name,
        ForceDeleteWithoutRecovery=True
    )
    print("Secret deletion initiated successfully!")
    print(f"Secret Name: {response['Name']}")
    print(f"Deletion Date: {response['DeletionDate']}")
except ClientError as e:
    print(f"Error deleting secret: {e}")

In [ ]:
# Delete the Security Group
try:
    ec2.delete_security_group(GroupId=security_group_id)
    print(f"Security Group '{security_group_name}' deleted successfully!")
except ClientError as e:
    print(f"Error deleting Security Group: {e}")

In [ ]:
# Delete the DB Subnet Group
try:
    rds.delete_db_subnet_group(DBSubnetGroupName=db_subnet_group_name)
    print(f"DB Subnet Group '{db_subnet_group_name}' deleted successfully!")
except ClientError as e:
    print(f"Error deleting DB Subnet Group: {e}")

Congratulations! You now know how to launch and create and AWS RDS instance using Amazon's Boto3 Python library. The commands with the CLI are very similar.

## Bonus Reference: AWS CLI Equivalents

If you prefer using the AWS CLI instead of boto3, here are the equivalent commands for the main operations in this exercise:

**Configure AWS Credentials:**
```bash
aws configure
# Then enter your Access Key ID, Secret Access Key, and region
```

**Create a Secret in AWS Secrets Manager:**
```bash
aws secretsmanager create-secret \
    --name rds-postgres-credentials \
    --description "RDS PostgreSQL credentials for exercise" \
    --secret-string '{"username":"dbadmin","password":"YourPasswordHere"}'
```

**Create an RDS Instance:**
```bash
aws rds create-db-instance \
    --db-instance-identifier postgres-exercise-db \
    --db-instance-class db.t3.micro \
    --engine postgres \
    --engine-version 18.1 \
    --master-username dbadmin \
    --master-user-password YourPasswordHere \
    --allocated-storage 20 \
    --db-name exercisedb \
    --publicly-accessible \
    --backup-retention-period 0 \
    --no-multi-az \
    --storage-type gp2
```

**Check RDS Instance Status:**
```bash
aws rds describe-db-instances --db-instance-identifier postgres-exercise-db
```

**Retrieve Secret from Secrets Manager:**
```bash
aws secretsmanager get-secret-value --secret-id rds-postgres-credentials
```

**Delete RDS Instance:**
```bash
aws rds delete-db-instance \
    --db-instance-identifier postgres-exercise-db \
    --skip-final-snapshot
```

**Delete Secret:**
```bash
aws secretsmanager delete-secret \
    --secret-id rds-postgres-credentials \
    --force-delete-without-recovery
```